In [1]:
import pandas as pd

caminho_arquivo = '/app/data/raw/spotify_train.csv'

# Liste apenas as colunas essenciais para o seu modelo e o target
colunas_uteis = ['track_popularity', 'energy', 'valence', 'danceability', 'acousticness', 'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo']

# Carrega o dataset filtrando as colunas
df = pd.read_csv(caminho_arquivo, usecols=colunas_uteis)

print(f"Dataset carregado! Linhas e colunas: {df.shape}")

Dataset carregado! Linhas e colunas: (945883, 10)


In [2]:
import pandas as pd

caminho_arquivo = '/app/data/raw/spotify_train.csv'

# Carrega apenas as primeiras 50.000 linhas para testar o código sem estourar a memória
df = pd.read_csv(caminho_arquivo, nrows=50000)

df['is_hit'] = (df['track_popularity'] >= 50).astype(int)
print("Amostra carregada com sucesso! Shape:", df.shape)

Amostra carregada com sucesso! Shape: (50000, 84)


In [3]:
# Calcula a proporção em porcentagem da variável alvo
porcentagem_alvo = df['is_hit'].value_counts(normalize=True) * 100

print("Distribuição da variável 'is_hit':")
for valor, p in porcentagem_alvo.items():
    status = "Hit (>= 50)" if valor == 1 else "Não-Hit (< 50)"
    print(f"- {status} (Classe {valor}): {p:.2f}%")

Distribuição da variável 'is_hit':
- Não-Hit (< 50) (Classe 0): 94.77%
- Hit (>= 50) (Classe 1): 5.23%


Dados do RandomForest aqui: Teste 1, teste 2, teste 3 

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Definir os caminhos dos arquivos de treino e teste
caminho_treino = '/app/data/raw/spotify_train.csv'
caminho_teste = '/app/data/raw/spotify_test.csv'

# Lista das features acústicas + a coluna de popularidade necessária para criar o 'is_hit'
colunas_necessarias = [
    'track_popularity', 'energy', 'valence', 'danceability', 'acousticness', 
    'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo'
]

print("Carregando dados de treino...")
df_train = pd.read_csv(caminho_treino, usecols=colunas_necessarias)
df_train['is_hit'] = (df_train['track_popularity'] >= 50).astype(int)

X_train = df_train[colunas_necessarias[1:]]
y_train = df_train['is_hit']

print("Carregando dados de teste...")
df_test = pd.read_csv(caminho_teste, usecols=colunas_necessarias)
df_test['is_hit'] = (df_test['track_popularity'] >= 50).astype(int)

X_test = df_test[colunas_necessarias[1:]]
y_test = df_test['is_hit']

# Limpar da memória RAM para evitar estouro
del df_train, df_test

# 2. Inicializar e treinar o Random Forest
modelo_rf = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)

print("\nTreinando o modelo Random Forest...")
modelo_rf.fit(X_train, y_train)

print("\nFazendo previsões no conjunto de teste...")
y_pred = modelo_rf.predict(X_test)

# 3. Extraindo os componentes da Matriz de Confusão
# tn: Verdadeiro Negativo | fp: Falso Positivo | fn: Falso Negativo | tp: Verdadeiro Positivo
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# 4. Exibindo as métricas detalhadas
print("\n" + "="*40)
print("📊 RELATÓRIO DE MÉTRICAS DETALHADAS")
print("="*40)

print(f"Acurácia Geral: {accuracy_score(y_test, y_pred)*100:.2f}%")

print("\n--- CONTAGEM DE ACERTOS E ERROS (Matriz de Confusão) ---")
print(f"✅ Acertos em Não-Hits (Verdadeiro Negativo - TN): {tn}")
print(f"❌ Falsos Positivos (Errou, achou que era hit mas não era - FP): {fp}")
print(f"❌ Falsos Negativos (Errou, deixou passar hits reais - FN): {fn}")
print(f"✅ Acertos em Hits (Verdadeiro Positivo - TP): {tp}")
print(f"📌 Total de acertos gerais: {tn + tp}")
print(f"📌 Total de erros gerais: {fp + fn}")

print("\n--- RELATÓRIO COMPLETO (F1-Score, Precision, Recall) ---")
print(classification_report(y_test, y_pred, target_names=['Não-Hit (0)', 'Hit (1)']))

Carregando dados de treino...
Carregando dados de teste...

Treinando o modelo Random Forest...

Fazendo previsões no conjunto de teste...

📊 RELATÓRIO DE MÉTRICAS DETALHADAS
Acurácia Geral: 94.96%

--- CONTAGEM DE ACERTOS E ERROS (Matriz de Confusão) ---
✅ Acertos em Não-Hits (Verdadeiro Negativo - TN): 112135
❌ Falsos Positivos (Errou, achou que era hit mas não era - FP): 170
❌ Falsos Negativos (Errou, deixou passar hits reais - FN): 5786
✅ Acertos em Hits (Verdadeiro Positivo - TP): 145
📌 Total de acertos gerais: 112280
📌 Total de erros gerais: 5956

--- RELATÓRIO COMPLETO (F1-Score, Precision, Recall) ---
              precision    recall  f1-score   support

 Não-Hit (0)       0.95      1.00      0.97    112305
     Hit (1)       0.46      0.02      0.05      5931

    accuracy                           0.95    118236
   macro avg       0.71      0.51      0.51    118236
weighted avg       0.93      0.95      0.93    118236



In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Definir os caminhos dos arquivos de treino e teste
caminho_treino = '/app/data/raw/spotify_train.csv'
caminho_teste = '/app/data/raw/spotify_test.csv'

# Mantemos apenas as colunas essenciais (track_popularity só para criar o is_hit, e as features acústicas/técnicas)
colunas_necessarias = [
    'track_popularity', 'energy', 'valence', 'danceability', 'acousticness', 
    'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo'
]

print("Carregando dados de treino...")
df_train = pd.read_csv(caminho_treino, usecols=colunas_necessarias)
df_train['is_hit'] = (df_train['track_popularity'] >= 50).astype(int)

X_train = df_train[colunas_necessarias[1:]]
y_train = df_train['is_hit']

print("Carregando dados de teste...")
df_test = pd.read_csv(caminho_teste, usecols=colunas_necessarias)
df_test['is_hit'] = (df_test['track_popularity'] >= 50).astype(int)

X_test = df_test[colunas_necessarias[1:]]
y_test = df_test['is_hit']

# Limpar da memória RAM
del df_train, df_test

# 2. Inicializar o Random Forest COM class_weight='balanced'
# Isso obriga o Random Forest a olhar para a classe minoritária (os hits)
modelo_rf = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', # <-- O segredo para não ignorar os hits
    random_state=42,
    n_jobs=-1
)

print("\nTreinando o Random Forest com balanceamento...")
modelo_rf.fit(X_train, y_train)

# 3. Em vez de usar o .predict() padrão (que usa limiar de 0.5), 
# vamos pegar as probabilidades para avaliarmos o comportamento
probabilidades = modelo_rf.predict_proba(X_test)[:, 1]

# Aqui você pode testar mudar o limiar (threshold). 
# O padrão é 0.5. Se abaixarmos para 0.4 ou 0.3, o modelo fica mais "ousado" em achar hits.
limiar = 0.4
y_pred_personalizado = (probabilidades >= limiar).astype(int)

# 4. Extraindo os componentes da Matriz de Confusão com o novo limiar
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_personalizado).ravel()

# 5. Exibindo as métricas detalhadas
print("\n" + "="*40)
print(f"📊 RANDOM FOREST (Com Limiar = {limiar})")
print("="*40)

print(f"Acurácia Geral: {accuracy_score(y_test, y_pred_personalizado)*100:.2f}%")

print("\n--- CONTAGEM DE ACERTOS E ERROS (Matriz de Confusão) ---")
print(f"✅ Acertos em Não-Hits (Verdadeiro Negativo - TN): {tn}")
print(f"❌ Falsos Positivos (Errou, achou que era hit mas não era - FP): {fp}")
print(f"❌ Falsos Negativos (Errou, deixou passar hits reais - FN): {fn}")
print(f"✅ Acertos em Hits (Verdadeiro Positivo - TP): {tp}")

print("\n--- RELATÓRIO COMPLETO ---")
print(classification_report(y_test, y_pred_personalizado, target_names=['Não-Hit (0)', 'Hit (1)']))

Carregando dados de treino...
Carregando dados de teste...

Treinando o Random Forest com balanceamento...

📊 RANDOM FOREST (Com Limiar = 0.4)
Acurácia Geral: 94.93%

--- CONTAGEM DE ACERTOS E ERROS (Matriz de Confusão) ---
✅ Acertos em Não-Hits (Verdadeiro Negativo - TN): 112010
❌ Falsos Positivos (Errou, achou que era hit mas não era - FP): 295
❌ Falsos Negativos (Errou, deixou passar hits reais - FN): 5705
✅ Acertos em Hits (Verdadeiro Positivo - TP): 226

--- RELATÓRIO COMPLETO ---
              precision    recall  f1-score   support

 Não-Hit (0)       0.95      1.00      0.97    112305
     Hit (1)       0.43      0.04      0.07      5931

    accuracy                           0.95    118236
   macro avg       0.69      0.52      0.52    118236
weighted avg       0.93      0.95      0.93    118236



In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Definir os caminhos dos arquivos de treino e teste
caminho_treino = '/app/data/raw/spotify_train.csv'
caminho_teste = '/app/data/raw/spotify_test.csv'

# Lista das features acústicas + track_popularity para criar o 'is_hit'
colunas_necessarias = [
    'track_popularity', 'energy', 'valence', 'danceability', 'acousticness', 
    'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo'
]

print("Carregando dados de treino...")
df_train = pd.read_csv(caminho_treino, usecols=colunas_necessarias)
df_train['is_hit'] = (df_train['track_popularity'] >= 50).astype(int)

X_train = df_train[colunas_necessarias[1:]]
y_train = df_train['is_hit']

print("Carregando dados de teste...")
df_test = pd.read_csv(caminho_teste, usecols=colunas_necessarias)
df_test['is_hit'] = (df_test['track_popularity'] >= 50).astype(int)

X_test = df_test[colunas_necessarias[1:]]
y_test = df_test['is_hit']

# Limpar da memória RAM para evitar estouro
del df_train, df_test

# 2. Inicializar e treinar o Random Forest com balanceamento
modelo_rf = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)

print("\nTreinando o modelo Random Forest...")
modelo_rf.fit(X_train, y_train)

# 3. Definir o Limiar (Threshold) personalizado
# Se 0.5 for muito restritivo, você pode abaixar aqui (ex: 0.3 ou 0.2) para o modelo arriscar mais
limiar = 0.25  # <-- Sinta-se livre para alterar este valor e testar

print(f"\nFazendo previsões no conjunto de teste usando limiar = {limiar}...")
probabilidades = modelo_rf.predict_proba(X_test)[:, 1]
y_pred_personalizado = (probabilidades >= limiar).astype(int)

# 4. Extraindo os componentes da Matriz de Confusão
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_personalizado).ravel()

# 5. Exibindo as métricas detalhadas
print("\n" + "="*45)
print(f"📊 RELATÓRIO RANDOM FOREST (Limiar = {limiar})")
print("="*45)

print(f"Acurácia Geral: {accuracy_score(y_test, y_pred_personalizado)*100:.2f}%")

print("\n--- CONTAGEM DE ACERTOS E ERROS (Matriz de Confusão) ---")
print(f"✅ Acertos em Não-Hits (Verdadeiro Negativo - TN): {tn}")
print(f"❌ Falsos Positivos (Errou, achou que era hit mas não era - FP): {fp}")
print(f"❌ Falsos Negativos (Errou, deixou passar hits reais - FN): {fn}")
print(f"✅ Acertos em Hits (Verdadeiro Positivo - TP): {tp}")
print(f"📌 Total de acertos gerais: {tn + tp}")
print(f"📌 Total de erros gerais: {fp + fn}")

print("\n--- RELATÓRIO COMPLETO (F1-Score, Precision, Recall) ---")
print(classification_report(y_test, y_pred_personalizado, target_names=['Não-Hit (0)', 'Hit (1)']))

Carregando dados de treino...
Carregando dados de teste...

Treinando o modelo Random Forest...

Fazendo previsões no conjunto de teste usando limiar = 0.25...

📊 RELATÓRIO RANDOM FOREST (Limiar = 0.25)
Acurácia Geral: 94.20%

--- CONTAGEM DE ACERTOS E ERROS (Matriz de Confusão) ---
✅ Acertos em Não-Hits (Verdadeiro Negativo - TN): 110723
❌ Falsos Positivos (Errou, achou que era hit mas não era - FP): 1582
❌ Falsos Negativos (Errou, deixou passar hits reais - FN): 5270
✅ Acertos em Hits (Verdadeiro Positivo - TP): 661
📌 Total de acertos gerais: 111384
📌 Total de erros gerais: 6852

--- RELATÓRIO COMPLETO (F1-Score, Precision, Recall) ---
              precision    recall  f1-score   support

 Não-Hit (0)       0.95      0.99      0.97    112305
     Hit (1)       0.29      0.11      0.16      5931

    accuracy                           0.94    118236
   macro avg       0.62      0.55      0.57    118236
weighted avg       0.92      0.94      0.93    118236



In [9]:
!pip install xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 9.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.5/351.5 MB 2.0 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


XGBOOST TESTES 

In [2]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Definir os caminhos dos arquivos de treino e teste
caminho_treino = '/app/data/raw/spotify_train.csv'
caminho_teste = '/app/data/raw/spotify_test.csv'

# Lista das features acústicas + a coluna de popularidade necessária para criar o 'is_hit'
colunas_necessarias = [
    'track_popularity', 'energy', 'valence', 'danceability', 'acousticness', 
    'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo'
]

print("Carregando dados de treino...")
df_train = pd.read_csv(caminho_treino, usecols=colunas_necessarias)
df_train['is_hit'] = (df_train['track_popularity'] >= 50).astype(int)

X_train = df_train[colunas_necessarias[1:]]
y_train = df_train['is_hit']

print("Carregando dados de teste...")
df_test = pd.read_csv(caminho_teste, usecols=colunas_necessarias)
df_test['is_hit'] = (df_test['track_popularity'] >= 50).astype(int)

X_test = df_test[colunas_necessarias[1:]]
y_test = df_test['is_hit']

# Limpar da memória RAM
del df_train, df_test

# 2. Calcular o peso para balancear as classes no XGBoost
# Isso diz ao modelo para dar mais importância à classe minoritária (hits)
negativos = (y_train == 0).sum()
positivos = (y_train == 1).sum()
proporcao_peso = negativos / positivos

# 3. Inicializar e configurar o XGBoost
modelo_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=proporcao_peso, # Essencial para dados desbalanceados!
    random_state=42,
    n_jobs=-1
)

print("\nTreinando o modelo XGBoost...")
modelo_xgb.fit(X_train, y_train)

print("\nFazendo previsões no conjunto de teste...")
y_pred = modelo_xgb.predict(X_test)

# 4. Extraindo os componentes da Matriz de Confusão
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# 5. Exibindo as métricas detalhadas
print("\n" + "="*40)
print("📊 RELATÓRIO DO XGBOOST - MÉTRICAS DETALHADAS")
print("="*40)

print(f"Acurácia Geral: {accuracy_score(y_test, y_pred)*100:.2f}%")

print("\n--- CONTAGEM DE ACERTOS E ERROS (Matriz de Confusão) ---")
print(f"✅ Acertos em Não-Hits (Verdadeiro Negativo - TN): {tn}")
print(f"❌ Falsos Positivos (Errou, achou que era hit mas não era - FP): {fp}")
print(f"❌ Falsos Negativos (Errou, deixou passar hits reais - FN): {fn}")
print(f"✅ Acertos em Hits (Verdadeiro Positivo - TP): {tp}")
print(f"📌 Total de acertos gerais: {tn + tp}")
print(f"📌 Total de erros gerais: {fp + fn}")

print("\n--- RELATÓRIO COMPLETO (F1-Score, Precision, Recall) ---")
print(classification_report(y_test, y_pred, target_names=['Não-Hit (0)', 'Hit (1)']))

Carregando dados de treino...
Carregando dados de teste...

Treinando o modelo XGBoost...

Fazendo previsões no conjunto de teste...

📊 RELATÓRIO DO XGBOOST - MÉTRICAS DETALHADAS
Acurácia Geral: 68.40%

--- CONTAGEM DE ACERTOS E ERROS (Matriz de Confusão) ---
✅ Acertos em Não-Hits (Verdadeiro Negativo - TN): 76425
❌ Falsos Positivos (Errou, achou que era hit mas não era - FP): 35880
❌ Falsos Negativos (Errou, deixou passar hits reais - FN): 1481
✅ Acertos em Hits (Verdadeiro Positivo - TP): 4450
📌 Total de acertos gerais: 80875
📌 Total de erros gerais: 37361

--- RELATÓRIO COMPLETO (F1-Score, Precision, Recall) ---
              precision    recall  f1-score   support

 Não-Hit (0)       0.98      0.68      0.80    112305
     Hit (1)       0.11      0.75      0.19      5931

    accuracy                           0.68    118236
   macro avg       0.55      0.72      0.50    118236
weighted avg       0.94      0.68      0.77    118236



In [11]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Definir os caminhos dos arquivos de treino e teste
caminho_treino = '/app/data/raw/spotify_train.csv'
caminho_teste = '/app/data/raw/spotify_test.csv'

# Lista das features acústicas + a coluna de popularidade necessária para criar o 'is_hit'
colunas_necessarias = [
    'track_popularity', 'energy', 'valence', 'danceability', 'acousticness', 
    'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo'
]

print("Carregando dados de treino...")
df_train = pd.read_csv(caminho_treino, usecols=colunas_necessarias)
df_train['is_hit'] = (df_train['track_popularity'] >= 50).astype(int)

X_train = df_train[colunas_necessarias[1:]]
y_train = df_train['is_hit']

print("Carregando dados de teste...")
df_test = pd.read_csv(caminho_teste, usecols=colunas_necessarias)
df_test['is_hit'] = (df_test['track_popularity'] >= 50).astype(int)

X_test = df_test[colunas_necessarias[1:]]
y_test = df_test['is_hit']

# Limpar da memória RAM
del df_train, df_test

# 2. Calcular o peso base e ajustar (suavizar o peso para não exagerar nos falsos positivos)
negativos = (y_train == 0).sum()
positivos = (y_train == 1).sum()
proporcao_peso = (negativos / positivos) * 0.7  # Ajustado para equilibrar melhor a precisão

# 3. Inicializar e configurar o XGBoost com parâmetros mais robustos
modelo_xgb = XGBClassifier(
    n_estimators=200,          # Aumentamos para dar mais capacidade de aprendizado
    learning_rate=0.05,        # Taxa de aprendizado menor para aprender com mais calma
    max_depth=7,               # Árvores um pouco mais profundas
    scale_pos_weight=proporcao_peso, 
    subsample=0.8,             # Usa 80% dos dados por árvore (evita overfitting)
    colsample_bytree=0.8,      # Usa 80% das features por árvore
    random_state=42,
    n_jobs=-1
)

print("\nTreinando o modelo XGBoost Otimizado...")
modelo_xgb.fit(X_train, y_train)

# 4. Definir o Limiar (Threshold) personalizado para o XGBoost
# O padrão é 0.5. Se quiser pegar mais hits, teste baixar para 0.4 ou 0.45
limiar = 0.45  

print(f"\nFazendo previsões no conjunto de teste com limiar = {limiar}...")
probabilidades = modelo_xgb.predict_proba(X_test)[:, 1]
y_pred_personalizado = (probabilidades >= limiar).astype(int)

# 5. Extraindo os componentes da Matriz de Confusão
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_personalizado).ravel()

# 6. Exibindo as métricas detalhadas
print("\n" + "="*50)
print(f"📊 RELATÓRIO DO XGBOOST OTIMIZADO (Limiar = {limiar})")
print("="*50)

print(f"Acurácia Geral: {accuracy_score(y_test, y_pred_personalizado)*100:.2f}%")

print("\n--- CONTAGEM DE ACERTOS E ERROS (Matriz de Confusão) ---")
print(f"✅ Acertos em Não-Hits (Verdadeiro Negativo - TN): {tn}")
print(f"❌ Falsos Positivos (Errou, achou que era hit mas não era - FP): {fp}")
print(f"❌ Falsos Negativos (Errou, deixou passar hits reais - FN): {fn}")
print(f"✅ Acertos em Hits (Verdadeiro Positivo - TP): {tp}")
print(f"📌 Total de acertos gerais: {tn + tp}")
print(f"📌 Total de erros gerais: {fp + fn}")

print("\n--- RELATÓRIO COMPLETO (F1-Score, Precision, Recall) ---")
print(classification_report(y_test, y_pred_personalizado, target_names=['Não-Hit (0)', 'Hit (1)']))

Carregando dados de treino...
Carregando dados de teste...

Treinando o modelo XGBoost Otimizado...

Fazendo previsões no conjunto de teste com limiar = 0.45...

📊 RELATÓRIO DO XGBOOST OTIMIZADO (Limiar = 0.45)
Acurácia Geral: 72.82%

--- CONTAGEM DE ACERTOS E ERROS (Matriz de Confusão) ---
✅ Acertos em Não-Hits (Verdadeiro Negativo - TN): 81939
❌ Falsos Positivos (Errou, achou que era hit mas não era - FP): 30366
❌ Falsos Negativos (Errou, deixou passar hits reais - FN): 1773
✅ Acertos em Hits (Verdadeiro Positivo - TP): 4158
📌 Total de acertos gerais: 86097
📌 Total de erros gerais: 32139

--- RELATÓRIO COMPLETO (F1-Score, Precision, Recall) ---
              precision    recall  f1-score   support

 Não-Hit (0)       0.98      0.73      0.84    112305
     Hit (1)       0.12      0.70      0.21      5931

    accuracy                           0.73    118236
   macro avg       0.55      0.72      0.52    118236
weighted avg       0.94      0.73      0.80    118236

